# DINOv2 finetune evaluation

Standalone, comprehensive evaluation on the held-out **test** split. Loads the
**fine-tuned checkpoint** (not the gated pretrained weights), so this runs without
any Hugging Face access once training is done.

**Metrics** (at image, **eye** [primary], and patient [worst-eye] levels):
precision, recall (= sensitivity), specificity, F1, **AUROC** (per-class OvR + macro),
AUPRC, accuracy (QWK intentionally omitted for this experiment), full **confusion matrices**, plus the
binary **referable-DR (R2+)** view with an operating-point sweep.

**Prerequisite:** run `Finetune_DINOv2.ipynb` first so `outputs/finetune_dinov2/checkpoint-best.pth` exists.

In [1]:
# ---- locate project root (this notebook lives in evaluation/) ----
import os, sys, json
HERE = os.getcwd()
PROJECT = HERE if os.path.isdir(os.path.join(HERE, "pipeline")) else os.path.dirname(HERE)
os.chdir(PROJECT)
sys.path.insert(0, os.path.join(PROJECT, "pipeline"))
sys.path.insert(0, os.path.join(PROJECT, "RETFound_repo"))
print("project root:", PROJECT)

import numpy as np, torch, pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (roc_curve, auc, precision_recall_curve, average_precision_score,
                             roc_auc_score, classification_report, confusion_matrix)
import dr_train as T, dr_eval as E

CKPT = os.path.join(PROJECT, "outputs", "finetune_dinov2", "checkpoint-best.pth")
assert os.path.exists(CKPT), f"missing {CKPT} -- run Finetune_DINOv2.ipynb first"
RESULTS = os.path.join(PROJECT, "evaluation", "results_dinov2"); os.makedirs(RESULTS, exist_ok=True)
CLASS_NAMES = ["R0", "R1", "R2", "R3"]; NC = 4

# Test-time augmentation: softmax averaged over flip views. Measured effect on this
# dataset is marginal (eye-level image averaging already reduces variance): the 4-view
# set lifted QWK & referable-sensitivity slightly. Set to ["identity"] to disable.
TTA_VIEWS = ["identity", "hflip", "vflip", "hvflip"]
NO_QWK = True   # if True, QWK is dropped from the summary table and metrics.json
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

project root: /home/eth/Desktop/isaack/Retfound.V2
device: cuda


In [2]:
# ---- rebuild model architecture and load the FINE-TUNED weights ----
ckpt = torch.load(CKPT, map_location="cpu")
cfg = ckpt.get("config", {})
print("checkpoint from epoch", ckpt.get("epoch"),
      ("" if NO_QWK else f"| val_QWK {round(ckpt.get('val_qwk', float('nan')), 4)}"),
      "| val_macro_AUROC", round(ckpt.get("val_macro_auroc", float('nan')), 4),
      "| val_macro_sens", round(ckpt.get("val_macro_sensitivity", float('nan')), 4))
DATA_PATH = cfg.get("data_path", "outputs/dr_imagefolder_cache")
INPUT = cfg.get("input_size", 224)
EVAL_BS = 32 if INPUT <= 224 else 12   # smaller batch for high-res eval

args = T.make_args({**{
    "data_path": DATA_PATH, "nb_classes": NC, "input_size": INPUT,
    # backbone is read from the checkpoint's stored config so the arch matches
    # the weights (MAE patch16 vs DINOv2 patch14) -- otherwise load_state_dict
    # fails on pos_embed / patch_embed shape mismatch.
    "model": cfg.get("model", "RETFound_mae"),
    "model_arch": cfg.get("model_arch", "retfound_mae"),
    "finetune_id": "", "drop_path": cfg.get("drop_path", 0.2),
    "batch_size": EVAL_BS, "accum_iter": 1, "epochs": 1, "warmup_epochs": 0,
    "blr": 5e-3, "layer_decay": 0.65, "weight_decay": 0.05, "min_lr": 1e-6,
    "output_dir": RESULTS, "seed": 42, "num_workers": 10,
}})
model = T.build_model_arch(args)
missing, unexpected = model.load_state_dict(ckpt["model"], strict=False)
assert not missing, f"unexpected missing keys: {missing}"
model.to(device).eval()
print(f"loaded fine-tuned weights OK | input_size={INPUT} eval_batch={EVAL_BS}")

/tmp/ipykernel_12379/1728938867.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(CKPT, map_location="cpu")


checkpoint from epoch 6  | val_macro_AUROC 0.946 | val_macro_sens 0.7916


/home/eth/miniforge3/envs/retfound/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


loaded fine-tuned weights OK | input_size=224 eval_batch=32


In [3]:
# ---- predict on the TEST split (order aligned to dataset.samples) ----
(_, _, ds_te), (_, _, dl_te) = T.build_loaders(args, shuffle_train=False)
assert ds_te.class_to_idx == json.load(open("outputs/class_mapping.json"))["ordinal_class_to_index"]
test_paths = [p for p, _ in ds_te.samples]
y_true, y_prob = E.predict(model, dl_te, device, tta=TTA_VIEWS)   # softmax averaged over TTA views
y_pred = y_prob.argmax(1)
print(f"test images: {len(y_true)}  |  eyes: {len(set(E.parse_pid_eye(p) for p in test_paths))}"
      f"  |  TTA views: {TTA_VIEWS}")

/home/eth/Desktop/isaack/Retfound.V2/pipeline/dr_eval.py:58: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp):


test images: 1286  |  eyes: 584  |  TTA views: ['identity', 'hflip', 'vflip', 'hvflip']


In [4]:
# ---- full metric bundle at image / eye / patient levels (saved to results/metrics.json) ----
report = E.full_report(test_paths, y_true, y_prob, RESULTS)
if NO_QWK:   # strip QWK from the saved artifact for this experiment
    for _lv in report:
        for _k in ("qwk", "kappa_unweighted"):
            report[_lv].pop(_k, None)
    json.dump(report, open(os.path.join(RESULTS, "metrics.json"), "w"), indent=2)

rows = []
for lvl in ["image_level", "eye_level", "patient_level"]:
    r = report[lvl]; b = r["binary_referable"]
    row = {"level": lvl.replace("_level", ""), "n": r["n"], "accuracy": r["accuracy"],
           "macro_AUROC": r["macro_auroc_ovr"], "macro_AUPRC": r["macro_auprc"],
           "macro_precision": r["macro_precision"], "macro_recall(sens)": r["macro_sensitivity"],
           "macro_specificity": r["macro_specificity"], "macro_F1": r["macro_f1"],
           "referable_AUROC": b.get("auroc"), "referable_AUPRC": b.get("auprc"),
           "referable_sens": b.get("sensitivity"), "referable_spec": b.get("specificity")}
    if not NO_QWK:
        row = {**{"level": row["level"], "n": row["n"], "QWK": r["qwk"]}, **row}
    rows.append(row)
summary = pd.DataFrame(rows).set_index("level")
summary.to_csv(os.path.join(RESULTS, "summary_metrics.csv"))
summary.round(4)

,n,accuracy,macro_AUROC,macro_AUPRC,macro_precision,macro_recall(sens),macro_specificity,macro_F1,referable_AUROC,referable_AUPRC,referable_sens,referable_spec
level,,,,,,,,,,,,
image,1286,0.8064,0.9159,0.7347,0.6908,0.6924,0.9130,0.6884,0.9674,0.7684,0.7581,0.9630
eye,584,0.8305,0.9257,0.7483,0.7094,0.7142,0.9256,0.7070,0.9687,0.7824,0.7719,0.9677
patient,330,0.8121,0.9139,0.7182,0.7023,0.6976,0.9218,0.6975,0.9691,0.7982,0.7895,0.9623


In [5]:
# ---- per-class table (eye level): precision / recall / specificity / F1 / support ----
eye = report["eye_level"]
pc = pd.DataFrame(eye["per_class"]).T
pc.index = CLASS_NAMES
pc = pc[["precision", "recall", "specificity", "f1", "support"]]
pc.loc["macro"] = [eye["macro_precision"], eye["macro_sensitivity"],
                   eye["macro_specificity"], eye["macro_f1"], pc["support"].sum()]
print("EYE-LEVEL per-class metrics:")
pc.round(4)

EYE-LEVEL per-class metrics:


,precision,recall,specificity,f1,support
R0,0.8757,0.9520,0.8207,0.9122,333.0
R1,0.8261,0.6856,0.9282,0.7493,194.0
R2,0.6875,0.6286,0.9818,0.6567,35.0
R3,0.4483,0.5909,0.9715,0.5098,22.0
macro,0.7094,0.7142,0.9256,0.7070,584.0


In [6]:
# ---- sklearn classification_report (eye level), for a familiar view ----
yt_e, yp_e = E.aggregate(test_paths, y_true, y_prob, "eye")
print("EYE-LEVEL classification report\n")
print(classification_report(yt_e, yp_e.argmax(1), labels=list(range(NC)),
                            target_names=CLASS_NAMES, zero_division=0, digits=4))

EYE-LEVEL classification report

              precision    recall  f1-score   support

          R0     0.8757    0.9520    0.9122       333
          R1     0.8261    0.6856    0.7493       194
          R2     0.6875    0.6286    0.6567        35
          R3     0.4483    0.5909    0.5098        22

    accuracy                         0.8305       584
   macro avg     0.7094    0.7142    0.7070       584
weighted avg     0.8318    0.8305    0.8276       584



In [7]:
# ---- confusion matrices: counts + row-normalized (eye and patient) ----
def plot_cm(ax, y, p, title, norm):
    cm = confusion_matrix(y, p, labels=list(range(NC))).astype(float)
    disp = cm / np.clip(cm.sum(1, keepdims=True), 1, None) if norm else cm
    im = ax.imshow(disp, cmap="Blues", vmin=0, vmax=disp.max() if disp.max() else 1)
    for i in range(NC):
        for j in range(NC):
            txt = f"{disp[i,j]:.2f}" if norm else f"{int(cm[i,j])}"
            ax.text(j, i, txt, ha="center", va="center", fontsize=9,
                    color="white" if disp[i, j] > 0.5 * disp.max() else "black")
    ax.set_xticks(range(NC)); ax.set_yticks(range(NC))
    ax.set_xticklabels(CLASS_NAMES); ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(title)

yt_p, yp_p = E.aggregate(test_paths, y_true, y_prob, "patient")
fig, ax = plt.subplots(2, 2, figsize=(10, 9))
plot_cm(ax[0, 0], yt_e, yp_e.argmax(1), "Eye — counts", False)
plot_cm(ax[0, 1], yt_e, yp_e.argmax(1), "Eye — row-normalized", True)
plot_cm(ax[1, 0], yt_p, yp_p.argmax(1), "Patient — counts", False)
plot_cm(ax[1, 1], yt_p, yp_p.argmax(1), "Patient — row-normalized", True)
fig.tight_layout(); fig.savefig(os.path.join(RESULTS, "confusion_matrices.png"), dpi=150); plt.show()

/tmp/ipykernel_12379/1844232079.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(os.path.join(RESULTS, "confusion_matrices.png"), dpi=150); plt.show()


In [8]:
# ---- ROC curves: per-class one-vs-rest + macro + binary referable (eye level) ----
yt_oh = np.eye(NC)[yt_e]
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
for c in range(NC):
    if yt_oh[:, c].sum() == 0:
        continue
    fpr, tpr, _ = roc_curve(yt_oh[:, c], yp_e[:, c])
    ax[0].plot(fpr, tpr, label=f"{CLASS_NAMES[c]} (AUC={auc(fpr,tpr):.3f})")
cols = [c for c in range(NC) if yt_oh[:, c].sum() > 0]
macro_auc = roc_auc_score(yt_oh[:, cols], yp_e[:, cols], average="macro", multi_class="ovr")
ax[0].plot([0, 1], [0, 1], "k--", lw=.7)
ax[0].set_title(f"ROC per class (OvR) — eye | macro AUROC={macro_auc:.3f}")
ax[0].set_xlabel("1 - specificity"); ax[0].set_ylabel("sensitivity"); ax[0].legend(fontsize=8)

b_true = (yt_e >= 2).astype(int); b_score = yp_e[:, 2:].sum(1)
fpr, tpr, _ = roc_curve(b_true, b_score)
ax[1].plot(fpr, tpr, label=f"referable DR (AUROC={roc_auc_score(b_true,b_score):.3f})", color="C3")
ax[1].plot([0, 1], [0, 1], "k--", lw=.7)
ax[1].set_title("ROC — binary referable DR (R2+) — eye")
ax[1].set_xlabel("1 - specificity"); ax[1].set_ylabel("sensitivity"); ax[1].legend()
fig.tight_layout(); fig.savefig(os.path.join(RESULTS, "roc_curves.png"), dpi=150); plt.show()

/tmp/ipykernel_12379/3845487002.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(os.path.join(RESULTS, "roc_curves.png"), dpi=150); plt.show()


In [9]:
# ---- Precision-Recall curves: per-class + binary referable (eye level) ----
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
for c in range(NC):
    if yt_oh[:, c].sum() == 0:
        continue
    pr, rc, _ = precision_recall_curve(yt_oh[:, c], yp_e[:, c])
    ap = average_precision_score(yt_oh[:, c], yp_e[:, c])
    ax[0].plot(rc, pr, label=f"{CLASS_NAMES[c]} (AP={ap:.3f})")
ax[0].set_title("PR per class (OvR) — eye"); ax[0].set_xlabel("recall"); ax[0].set_ylabel("precision"); ax[0].legend(fontsize=8)

pr, rc, _ = precision_recall_curve(b_true, b_score)
ax[1].plot(rc, pr, color="C3", label=f"referable DR (AUPRC={average_precision_score(b_true,b_score):.3f})")
ax[1].axhline(b_true.mean(), ls="--", c="grey", lw=.7, label=f"prevalence={b_true.mean():.3f}")
ax[1].set_title("PR — binary referable DR (R2+) — eye"); ax[1].set_xlabel("recall (sensitivity)")
ax[1].set_ylabel("precision (PPV)"); ax[1].legend()
fig.tight_layout(); fig.savefig(os.path.join(RESULTS, "pr_curves.png"), dpi=150); plt.show()

/tmp/ipykernel_12379/300811277.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(os.path.join(RESULTS, "pr_curves.png"), dpi=150); plt.show()


In [10]:
# ---- per-class metric bar chart (eye level) ----
metrics = ["precision", "recall", "specificity", "f1"]
vals = {m: [eye["per_class"][c][m] for c in range(NC)] for m in metrics}
x = np.arange(NC); w = 0.2
fig, ax = plt.subplots(figsize=(9, 4.5))
for i, m in enumerate(metrics):
    ax.bar(x + (i - 1.5) * w, vals[m], w, label=m)
ax.set_xticks(x); ax.set_xticklabels(CLASS_NAMES); ax.set_ylim(0, 1.05)
ax.set_title("Per-class metrics (eye level)"); ax.set_ylabel("score"); ax.legend(ncol=4, fontsize=9)
for i, m in enumerate(metrics):
    for j in range(NC):
        v = vals[m][j]
        if v == v:
            ax.text(x[j] + (i - 1.5) * w, v + 0.01, f"{v:.2f}", ha="center", fontsize=7, rotation=90)
fig.tight_layout(); fig.savefig(os.path.join(RESULTS, "per_class_bars.png"), dpi=150); plt.show()

/tmp/ipykernel_12379/572145000.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(os.path.join(RESULTS, "per_class_bars.png"), dpi=150); plt.show()


In [11]:
# ---- operating-point sweep for referable DR (choose a sensitivity target) ----
from sklearn.metrics import confusion_matrix as cmat
rows = []
for thr in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    pred = (b_score >= thr).astype(int)
    tn, fp, fn, tp = cmat(b_true, pred, labels=[0, 1]).ravel()
    rows.append({"threshold": thr,
                 "sensitivity": tp / (tp + fn) if (tp + fn) else float("nan"),
                 "specificity": tn / (tn + fp) if (tn + fp) else float("nan"),
                 "precision(PPV)": tp / (tp + fp) if (tp + fp) else float("nan"),
                 "TP": tp, "FP": fp, "FN": fn, "TN": tn})
op = pd.DataFrame(rows).set_index("threshold")
op.to_csv(os.path.join(RESULTS, "referable_operating_points.csv"))
print("Referable-DR operating points (eye level):")
op.round(4)

Referable-DR operating points (eye level):


,sensitivity,specificity,precision(PPV),TP,FP,FN,TN
threshold,,,,,,,
0.2,0.9474,0.8824,0.4655,54,62,3,465
0.3,0.8947,0.9317,0.5862,51,36,6,491
0.4,0.8070,0.9488,0.6301,46,27,11,500
0.5,0.7719,0.9677,0.7213,44,17,13,510
0.6,0.7193,0.9791,0.7885,41,11,16,516
0.7,0.5789,0.9829,0.7857,33,9,24,518


## Notes on interpretation
- **Eye level is primary**; patient level uses worst-eye aggregation. Image level is shown for completeness.
- Minority classes are small in the test split (~35 R2, ~22 R3 eyes) — per-class precision/recall
  for R2/R3 and the referable-DR PR curve rest on few positives; treat them as indicative and prefer
  k-fold CV for a firm estimate.
- For screening, pick the operating point from the sweep above by the **sensitivity** you require for
  referable DR, and report the corresponding specificity/PPV — do not default to 0.5.
- **Test-time augmentation** (`TTA_VIEWS`, top cell) averages softmax over flip views. On this dataset
  the gain is marginal (eye-level image averaging already reduces variance); the 4-view set slightly
  improved QWK and referable-sensitivity. Set `["identity"]` to disable and compare.
- All figures and tables are saved under this notebook's results folder.